In [130]:
import pandas as pd
import json
import openpyxl

## PROCESSING OF THE EXCEL ##

In [131]:
name_file = "/Users/alexialozano/Documents/GitHub/IDTA-AAS-Fibro/data/xlsx/DPP_FIBROTOR_ER15_V2.xlsx"

wb = openpyxl.load_workbook(name_file, data_only=True)
print(wb.sheetnames)

['Digital Nameplate', 'Handover Documentation', 'Technical Data', 'Carbon Footprint', 'Maintenance Instructions', 'Sheet1']


## Obtaining the Templates ##

In [ ]:
import openpyxl
import json


def extract_table(rows, start):

    headers = [
        str(h).strip() if h else ""
        for h in rows[start]
    ]

    try:
        idshort_col = headers.index("Element Name (idShort)")
        value_col = headers.index("Actual Value")
        obligation_col = headers.index("Obligation")

    except ValueError:
        print("Invalid header:", headers)
        return {}, start + 1, []


    data = {}
    missing_mandatory = []

    i = start + 1

    while i < len(rows):

        row = rows[i]

        if "Element Name (idShort)" in row:
            break

        if any(cell is not None for cell in row):

            key = row[idshort_col]
            value = row[value_col]
            obligation = row[obligation_col]

            if key:

                data[key] = value

                if (
                    obligation == "Mandatory"
                    and (value is None or str(value).strip() == "")
                ):
                    missing_mandatory.append(key)

        i += 1


    return data, i, missing_mandatory



def find_submodels(sheet):

    submodels = {}
    missing = []

    rows = list(sheet.iter_rows(values_only=True))

    i = 0

    while i < len(rows):

        row = rows[i]

        if row and "Element Name (idShort)" in row:

            name = rows[i-1][0] if i > 0 else "Unknown"

            table, end, missing_fields = extract_table(rows, i)

            submodels[name] = table
            missing.extend(missing_fields)

            i = end

        else:
            i += 1


    return submodels, missing



wb = openpyxl.load_workbook(
    name_file,
    data_only=True
)


model = {}

for sheet in wb:
    model[sheet.title] = find_submodels(sheet)

flat = {}

for sheet in model.values():
    for table in sheet.values():
        flat.update(table)

with open(
    "output.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        flat,
        f,
        indent=4,
        ensure_ascii=False
    )

In [133]:
def collect_model_types(elements, types=None):
    if types is None:
        types = set()

    for element in elements:
        model = element.get("modelType")
        if model:
            types.add(model)

        if (
            model in ("SubmodelElementCollection", "SubmodelElementList")
            and isinstance(element.get("value"), list)
        ):
            collect_model_types(element["value"], types)

    return types

In [134]:
def fill_template(elements, company_data):

    for element in elements:

        key = element.get("idShort")
        model = element.get("modelType")

        print(f"\nProcessing {key} ({model})")

        if key in company_data:

            value = company_data[key]
            print("Excel value:", value)

            if model == "Property":
                element["value"] = value

            elif model == "MultiLanguageProperty":
                if element.get("value"):
                    element["value"][0]["text"] = value

            elif model == "File":
                element["value"] = value

        if (
            model == "SubmodelElementCollection"
            and isinstance(element.get("value"), list)
        ):
            fill_template(element["value"], company_data)

    return elements

In [135]:
import json


# Load template
with open("/Users/alexialozano/Documents/GitHub/IDTA-AAS-Fibro/data/xlsx/IDTA 02006-3-0-1_Template_Digital Nameplate.json") as f:
    aas = json.load(f)


# Load extracted Excel data
with open("output.json") as f:
    company = json.load(f)


submodel = aas["submodels"][0]

types = collect_model_types(submodel["submodelElements"])
print(types)

submodel["submodelElements"] = fill_template(
    submodel["submodelElements"],
    company
)


with open("DigitalNameplate_generated.json", "w") as f:
    json.dump(
        aas,
        f,
        indent=4
    )

{'SubmodelElementCollection', 'MultiLanguageProperty', 'SubmodelElementList', 'File', 'Property'}

Processing URIOfTheProduct (Property)
Excel value: https://www.fibrort.com/downloads?product=25

Processing ManufacturerName (MultiLanguageProperty)
Excel value: Fibro Rundtische GmbH@en

Processing ManufacturerProductDesignation (MultiLanguageProperty)
Excel value: FIBROTOR ER.15 indexing table@en

Processing AddressInformation (SubmodelElementCollection)
Excel value: Address information entered in rows 31–34.

Processing ManufacturerProductRoot (MultiLanguageProperty)
Excel value: None

Processing ManufacturerProductFamily (MultiLanguageProperty)
Excel value: None

Processing ManufacturerProductType (Property)
Excel value: None

Processing OrderCodeOfManufacturer (Property)
Excel value: ER.15.0410.1.142.04.0.0.1

Processing ProductArticleNumberOfManufacturer (Property)
Excel value: 4-815-911-0483

Processing SerialNumber (Property)
Excel value: 0024005004-010-01

Processing YearOfConstr

VALIDATION FOR IMPORTING INTO THE SHELL

In [136]:
from basyx.aas import model
from basyx.aas.adapter import json

with open("output.json", "r") as f:
    store = json.read_aas_json_file(f)

print("Model successfully loaded!")

Model successfully loaded!


In [ ]:
import json
from aas_core3 import jsonization

with open("DigitalNameplate_generated.json", "r", encoding="utf-8") as f:
    data = json.load(f)
try:
    environment = jsonization.environment_from_jsonable(data)
    print("✅ Valid AAS Environment")
except Exception as e:
    print("❌ Validation failed")
    print(e)

❌ Validation failed
Expected a str, but got: <class 'NoneType'>
